In [ ]:
!pip -q install yfinance ta scikit-learn pandas numpy


  Preparing metadata (setup.py) ... done


In [ ]:
from google.colab import drive
import os


drive.mount('/content/drive', force_remount=True)
data_dir = '/content/drive/MyDrive/Close_res/'
folder_path=data_dir
os.chdir('/content/drive/MyDrive/Close_res/')

Mounted at /content/drive


In [ ]:
# =========================
# 1) Install dependencies
# =========================

# =========================
# 2) Imports
# =========================
import itertools
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime

# technical indicators
from ta.trend import SMAIndicator, EMAIndicator, MACD, ADXIndicator
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

# selection
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

# =========================
# 3) Config
# =========================
START = "2010-01-01"
END = None  # to today

# Robust ticker candidates for each index (Yahoo Finance)
INDEX_TICKERS = {
    "SP500": ["^GSPC"],
    "EUROSTOXX50": ["^STOXX50E", "^SX5E"],
    "NIKKEI225": ["^N225"],
}

OHLCV = ["Open", "High", "Low", "Close", "Volume"]

# =========================
# 4) Helpers
# =========================

def download_first_working(candidates, start=None, end=None):
    """
    Try tickers in order; return (ticker, cleaned OHLCV DataFrame).
    Uses auto_adjust=True so Close always exists (Adj Close is dropped).
    """
    for t in candidates:
        df = yf.download(
            t, start=start, end=end,
            auto_adjust=True,            # <-- key change
            progress=False,
            group_by="column",
            multi_level_index=False,     # <-- keep columns flat
            threads=False,
            actions=False
        )
        if df is None or df.empty:
            continue

        # Ensure standard columns exist; Volume may be NaN/0 for some indices
        for col in ["Open","High","Low","Close","Volume"]:
            if col not in df.columns:
                df[col] = np.nan

        df = df[["Open","High","Low","Close","Volume"]].copy()
        df.index = pd.to_datetime(df.index)
        df = df.sort_index().dropna(subset=["Close"])

        if not df.empty:
            return t, df

    raise ValueError(f"No working ticker among: {candidates}")

def has_usable_volume(df: pd.DataFrame) -> bool:
    if "Volume" not in df.columns:
        return False
    v = df["Volume"]
    return not (v.isna().all() or (v.fillna(0) == 0).all())

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add indicators using standard parameters.
    SMA(5,20), EMA(12,26), MACD(12,26,9), RSI(14), Stoch(14,3),
    Boll(20,2), ADX(14), OBV (if volume), ROC(10), ATR(14)
    """
    out = df.copy()

    # Moving averages: keep windows small as requested
    out["sma_5"]  = SMAIndicator(close=out["Close"], window=5, fillna=False).sma_indicator()
    out["sma_20"] = SMAIndicator(close=out["Close"], window=20, fillna=False).sma_indicator()

    out["ema_12"] = EMAIndicator(close=out["Close"], window=12, fillna=False).ema_indicator()
    out["ema_26"] = EMAIndicator(close=out["Close"], window=26, fillna=False).ema_indicator()

    macd = MACD(close=out["Close"], window_slow=26, window_fast=12, window_sign=9, fillna=False)
    out["macd"]        = macd.macd()
    out["macd_signal"] = macd.macd_signal()
    out["macd_hist"]   = macd.macd_diff()

    out["rsi_14"] = RSIIndicator(close=out["Close"], window=14, fillna=False).rsi()

    stoch = StochasticOscillator(
        high=out["High"], low=out["Low"], close=out["Close"],
        window=14, smooth_window=3, fillna=False
    )
    out["stoch_k_14_3"] = stoch.stoch()
    out["stoch_d_14_3"] = stoch.stoch_signal()

    bb = BollingerBands(close=out["Close"], window=20, window_dev=2, fillna=False)
    out["bb_m_20_2"] = bb.bollinger_mavg()
    out["bb_h_20_2"] = bb.bollinger_hband()
    out["bb_l_20_2"] = bb.bollinger_lband()
    # Optional: bandwidth and %B
    out["bb_bw_20_2"] = (out["bb_h_20_2"] - out["bb_l_20_2"]) / out["bb_m_20_2"]
    # PercentB: position within bands
    out["bb_pctb_20_2"] = (out["Close"] - out["bb_l_20_2"]) / (out["bb_h_20_2"] - out["bb_l_20_2"])

    out["adx_14"] = ADXIndicator(
        high=out["High"], low=out["Low"], close=out["Close"], window=14, fillna=False
    ).adx()

    # OBV only if volume looks usable
    if has_usable_volume(out):
        out["obv"] = OnBalanceVolumeIndicator(close=out["Close"], volume=out["Volume"], fillna=False).on_balance_volume()
    else:
        out["obv"] = np.nan  # will be dropped later if all-NaN

    out["roc_10"] = ROCIndicator(close=out["Close"], window=10, fillna=False).roc()
    out["atr_14"] = AverageTrueRange(
        high=out["High"], low=out["Low"], close=out["Close"], window=14, fillna=False
    ).average_true_range()

    # Drop pure-NaN columns (e.g., OBV when volume unusable)
    out = out.loc[:, ~out.isna().all()]

    return out

def make_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add next-day targets (aligned so that features at t map to target at t+1)
    """
    out = df.copy()
    out["y_close_t+1"] = out["Close"].shift(-1)
    out["y_logret_t+1"] = np.log(out["Close"].shift(-1) / out["Close"])
    return out

def time_series_train_test_split(df: pd.DataFrame, train_frac=0.8):
    n = len(df)
    cut = int(n * train_frac)
    train = df.iloc[:cut].copy()
    test  = df.iloc[cut:].copy()
    return train, test

def corr_and_lasso_select(train_df: pd.DataFrame, base_cols, cand_cols, target_col,
                          top_k_corr=12, corr_threshold=0.02, collinear_threshold=0.95,
                          n_splits=5, random_state=0):
    """
    1) keep indicators with |corr(y)| above threshold,
       take top_k by absolute correlation (cap),
       and drop collinear ones (|corr|>collinear_threshold).
    2) run LassoCV (with TimeSeriesSplit) on standardized features to prune further.
    Always keep base_cols (OHLCV).
    Returns: list of selected indicator columns (not including base_cols).
    """
    # --- 1) correlation screen ---
    corr_vals = (
        train_df[cand_cols + [target_col]]
        .dropna()
        .corr()[target_col]
        .drop(labels=[target_col])
        .abs()
        .sort_values(ascending=False)
    )
    corr_keep = corr_vals[corr_vals >= corr_threshold].index.tolist()
    corr_keep = corr_keep[:top_k_corr] if top_k_corr is not None else corr_keep
    if not corr_keep:
        return []  # nothing passes minimum signal

    # drop collinear features among corr_keep
    subset = train_df[corr_keep].dropna()
    to_drop = set()
    if subset.shape[1] > 1:
        cm = subset.corr().abs()
        # upper triangle indices
        for i in range(len(cm.columns)):
            for j in range(i+1, len(cm.columns)):
                if cm.iloc[i, j] > collinear_threshold:
                    # drop less correlated-to-target one
                    ci, cj = cm.columns[i], cm.columns[j]
                    if corr_vals[ci] >= corr_vals[cj]:
                        to_drop.add(cj)
                    else:
                        to_drop.add(ci)
    pruned = [c for c in corr_keep if c not in to_drop]

    # --- 2) LassoCV on pruned set ---
    if not pruned:
        return []

    X = train_df[base_cols + pruned]
    y = train_df[target_col]
    tmp = pd.concat([X, y], axis=1).dropna()
    X, y = tmp[base_cols + pruned], tmp[target_col]

    if len(tmp) < 100 or len(pruned) == 0:
        # not enough data for CV; accept pruned list
        return pruned

    tscv = TimeSeriesSplit(n_splits=min(n_splits, len(tmp)//50 if len(tmp)//50 >= 3 else 3))
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("lasso",  LassoCV(alphas=np.logspace(-4, 2, 50), cv=tscv, max_iter=20000, random_state=random_state)),
    ])
    pipe.fit(X, y)

    # coefficients for indicators only (exclude base OHLCV from selection)
    coef = pipe.named_steps["lasso"].coef_
    feat_names = X.columns.tolist()
    # map feature -> coef
    keep = []
    for name, b in zip(feat_names, coef):
        if name in pruned and np.abs(b) > 1e-8:
            keep.append(name)

    # fall back to pruned if Lasso zeros everything
    return keep if keep else pruned

def build_and_save_scenarios(index_name, df_raw):
    """
    For one index:
      - compute indicators
      - add targets
      - build Scenario A/B/C/D dataframes (drop initial NaNs from rolling indicators)
      - save to CSV
    """
    df = compute_indicators(df_raw)
    df = make_targets(df)

    # --- define columns ---
    base_ohlcv = [c for c in OHLCV if c in df.columns]  # Volume may be present or not
    indicator_cols = [c for c in df.columns if c not in (["Adj Close"] + base_ohlcv + ["y_close_t+1","y_logret_t+1"])]

    # Drop rows that have NaNs produced by indicator warmups
    df_clean = df.dropna(subset=["y_close_t+1","y_logret_t+1"]).copy()
    # Be conservative: drop rows where any selected indicator NaN
    df_clean = df_clean.dropna(subset=list(set(indicator_cols)))  # if all volume-based features are NaN, they were removed upstream

    # --- Scenario A: Close only ---
    A = df_clean[["Close", "y_close_t+1", "y_logret_t+1"]].copy()
    A.to_csv(f"{index_name}_scenario_A.csv")

    # --- Scenario B: OHLCV ---
    B_cols = list(dict.fromkeys(base_ohlcv + ["y_close_t+1", "y_logret_t+1"]))
    B = df_clean[B_cols].copy()
    B.to_csv(f"{index_name}_scenario_B.csv")

    # --- Scenario C: OHLCV + ALL indicators ---
    C_cols = list(dict.fromkeys(base_ohlcv + indicator_cols + ["y_close_t+1", "y_logret_t+1"]))
    C = df_clean[C_cols].copy()
    C.to_csv(f"{index_name}_scenario_C.csv")

    # --- Scenario D: OHLCV + FILTERED indicators (train-only selection) ---
    # Split train/test chronologically for selection (80/20)
    train_df, test_df = time_series_train_test_split(df_clean, train_frac=0.8)

    selected_inds = corr_and_lasso_select(
        train_df=train_df,
        base_cols=base_ohlcv,
        cand_cols=indicator_cols,
        target_col="y_logret_t+1",     # selection on next-day log return (stationary-ish)
        top_k_corr=12,
        corr_threshold=0.02,
        collinear_threshold=0.95,
        n_splits=5,
        random_state=0
    )
    D_cols = list(dict.fromkeys(base_ohlcv + selected_inds + ["y_close_t+1", "y_logret_t+1"]))
    D = df_clean[D_cols].copy()
    D.to_csv(f"{index_name}_scenario_D.csv")

    print(f"\n[{index_name}] usable rows: {len(df_clean):,}")
    print(f"[{index_name}] OHLCV columns used: {base_ohlcv}")
    print(f"[{index_name}] Indicators computed ({len(indicator_cols)}): {indicator_cols}")
    print(f"[{index_name}] FILTERED indicators kept for Scenario D ({len(selected_inds)}): {selected_inds}")
    print(f"[{index_name}] Saved CSVs -> {index_name}_scenario_[A|B|C|D].csv")

# =========================
# 5) Run for all indices
# =========================
results = {}
for name, candidates in INDEX_TICKERS.items():
    ticker, df_raw = download_first_working(candidates, start=START, end=END)
    print(f"{name}: using ticker {ticker}, rows={len(df_raw):,}, date range {df_raw.index.min().date()} → {df_raw.index.max().date()}")
    build_and_save_scenarios(name, df_raw)


SP500: using ticker ^GSPC, rows=3,933, date range 2010-01-04 → 2025-08-21

[SP500] usable rows: 3,899
[SP500] OHLCV columns used: ['Open', 'High', 'Low', 'Close', 'Volume']
[SP500] Indicators computed (19): ['sma_5', 'sma_20', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_hist', 'rsi_14', 'stoch_k_14_3', 'stoch_d_14_3', 'bb_m_20_2', 'bb_h_20_2', 'bb_l_20_2', 'bb_bw_20_2', 'bb_pctb_20_2', 'adx_14', 'obv', 'roc_10', 'atr_14']
[SP500] FILTERED indicators kept for Scenario D (2): ['roc_10', 'rsi_14']
[SP500] Saved CSVs -> SP500_scenario_[A|B|C|D].csv
EUROSTOXX50: using ticker ^STOXX50E, rows=3,922, date range 2010-01-04 → 2025-08-21

[EUROSTOXX50] usable rows: 3,888
[EUROSTOXX50] OHLCV columns used: ['Open', 'High', 'Low', 'Close', 'Volume']
[EUROSTOXX50] Indicators computed (19): ['sma_5', 'sma_20', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_hist', 'rsi_14', 'stoch_k_14_3', 'stoch_d_14_3', 'bb_m_20_2', 'bb_h_20_2', 'bb_l_20_2', 'bb_bw_20_2', 'bb_pctb_20_2', 'adx_14', 'obv', 'roc_

In [ ]:
# =========================================
# Leakage-safe normalization + time splits
# =========================================
!pip -q install pandas numpy scikit-learn

import os, json, math
import numpy as np
import pandas as pd

# --------- Config ---------
BASE_DIR = ""                    # where your scenario CSVs are
OUT_DIR  = "/content/drive/MyDrive/Close_res/prepared"
INDEXES  = ["SP500", "EUROSTOXX50", "NIKKEI225"]
SCENARIOS = ["A", "B", "C", "D"]

WINDOW = 60                              # rolling window length for μ/σ
TRAIN_FRAC = 0.70                        # chronological split
VAL_FRAC   = 0.10                        # test implicitly = 1 - TRAIN - VAL

TARGET_COLS = ["y_close_t+1", "y_logret_t+1"]   # kept raw (not normalized)

os.makedirs(OUT_DIR, exist_ok=True)

def _ensure_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing input: {path} — run the earlier feature script first.")

def _rolling_norm_no_leak(df, feature_cols, window=60):
    """
    Leakage-safe rolling z-score:
      z_t = (x_t - mean_{t-1..t-W}) / std_{t-1..t-W}
    Returns:
      z_df  : normalized dataframe with original target columns preserved
      mu_df : rolling means aligned to z_df.index (columns 'mu_<feature>')
      sd_df : rolling stds   aligned to z_df.index (columns 'sd_<feature>')
    """
    # compute μ and σ from strictly prior data
    mu = df[feature_cols].shift(1).rolling(window=window, min_periods=window).mean()
    sd = df[feature_cols].shift(1).rolling(window=window, min_periods=window).std(ddof=0)

    # avoid divide-by-zero
    sd_repl = sd.replace(0, np.nan)

    z = (df[feature_cols] - mu) / sd_repl

    # Align to dates where all features have valid z AND targets exist
    valid_idx = z.dropna().index
    if TARGET_COLS:
        valid_idx = valid_idx.intersection(df.dropna(subset=TARGET_COLS).index)

    z = z.loc[valid_idx]
    mu = mu.loc[valid_idx]
    sd = sd.loc[valid_idx]

    # Rename μ/σ columns for saving
    mu.columns = [f"mu_{c}" for c in mu.columns]
    sd.columns = [f"sd_{c}" for c in sd.columns]

    # Reattach targets (unscaled)
    out = pd.concat([z, df.loc[valid_idx, TARGET_COLS]], axis=1)

    return out, mu, sd

def _chronological_splits(df, train_frac=0.7, val_frac=0.1):
    n = len(df)
    assert 0 < train_frac < 1 and 0 <= val_frac < 1 and train_frac + val_frac < 1, "Bad split fractions."
    i_train_end = int(math.floor(n * train_frac))
    i_val_end   = int(math.floor(n * (train_frac + val_frac)))
    train = df.iloc[:i_train_end].copy()
    val   = df.iloc[i_train_end:i_val_end].copy()
    test  = df.iloc[i_val_end:].copy()
    return train, val, test

def _save_split_frames(base_path, df_full, mu_df, sd_df, feature_cols, target_cols):
    os.makedirs(base_path, exist_ok=True)

    # Save normalized full dataset
    df_full.to_csv(os.path.join(base_path, "normalized_full.csv.gz"), index=True)

    # Save rolling μ/σ per timestamp
    roll_stats = pd.concat([mu_df, sd_df], axis=1)
    roll_stats.to_csv(os.path.join(base_path, "rolling_stats.csv.gz"), index=True)

    # Chronological splits
    train, val, test = _chronological_splits(df_full, TRAIN_FRAC, VAL_FRAC)

    # Feature/target lists
    meta = {
        "feature_cols": feature_cols,
        "target_cols": target_cols,
        "window": WINDOW,
        "splits": {
            "train": {"rows": len(train), "start": str(train.index.min()), "end": str(train.index.max()) if len(train) else None},
            "val":   {"rows": len(val), "start": str(val.index.min()),     "end": str(val.index.max()) if len(val) else None},
            "test":  {"rows": len(test), "start": str(test.index.min()),    "end": str(test.index.max()) if len(test) else None},
        }
    }

    # Train-only global μ/σ snapshot (not used for z_t, but handy to keep)
    if len(train):
        train_mu = train[feature_cols].mean(numeric_only=True)
        train_sd = train[feature_cols].std(ddof=0, numeric_only=True).replace(0, np.nan)
        meta["train_global_mu"] = {c: (None if pd.isna(v) else float(v)) for c, v in train_mu.items()}
        meta["train_global_sd"] = {c: (None if pd.isna(v) else float(v)) for c, v in train_sd.items()}

        # Also save as a separate JSON for quick reuse
        with open(os.path.join(base_path, "train_scaler.json"), "w") as f:
            json.dump({
                "mu": meta["train_global_mu"],
                "sd": meta["train_global_sd"],
                "window": WINDOW
            }, f, indent=2)

    # Save splits
    train.to_csv(os.path.join(base_path, "train.csv.gz"))
    val.to_csv(os.path.join(base_path, "val.csv.gz"))
    test.to_csv(os.path.join(base_path, "test.csv.gz"))

    # Save meta
    with open(os.path.join(base_path, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    return len(train), len(val), len(test)

# --------------- Run ---------------
for idx in INDEXES:
    for sc in SCENARIOS:
        in_path = f"{idx}_scenario_{sc}.csv"
        _ensure_exists(in_path)
        df = pd.read_csv(in_path, parse_dates=[0], index_col=0)

        # Identify features to normalize (everything except targets)
        cols = df.columns.tolist()
        feature_cols = [c for c in cols if c not in TARGET_COLS]

        # Apply leakage-safe rolling normalization
        z_df, mu_df, sd_df = _rolling_norm_no_leak(df, feature_cols, window=WINDOW)

        # Save normalized full + splits + rolling stats + meta
        base_out = f"{OUT_DIR}/{idx}/scenario_{sc}"
        ntr, nv, nts = _save_split_frames(base_out, z_df, mu_df, sd_df, feature_cols, TARGET_COLS)

        print(f"[{idx} - {sc}] rows after warm-up drop: {len(z_df):,}  | train={ntr:,} val={nv:,} test={nts:,}")
        print(f"  -> Saved to: {base_out}")


[SP500 - A] rows after warm-up drop: 3,839  | train=2,687 val=384 test=768
  -> Saved to: /content/drive/MyDrive/Close_res/prepared/SP500/scenario_A
[SP500 - B] rows after warm-up drop: 3,839  | train=2,687 val=384 test=768
  -> Saved to: /content/drive/MyDrive/Close_res/prepared/SP500/scenario_B
[SP500 - C] rows after warm-up drop: 3,839  | train=2,687 val=384 test=768
  -> Saved to: /content/drive/MyDrive/Close_res/prepared/SP500/scenario_C
[SP500 - D] rows after warm-up drop: 3,839  | train=2,687 val=384 test=768
  -> Saved to: /content/drive/MyDrive/Close_res/prepared/SP500/scenario_D
[EUROSTOXX50 - A] rows after warm-up drop: 3,828  | train=2,679 val=383 test=766
  -> Saved to: /content/drive/MyDrive/Close_res/prepared/EUROSTOXX50/scenario_A
[EUROSTOXX50 - B] rows after warm-up drop: 3,023  | train=2,116 val=302 test=605
  -> Saved to: /content/drive/MyDrive/Close_res/prepared/EUROSTOXX50/scenario_B
[EUROSTOXX50 - C] rows after warm-up drop: 3,023  | train=2,116 val=302 test=605
 

In [ ]:
# ================================================================
# TRAINING (fixed): predict z-target (causal), report raw metrics
# ================================================================
!pip -q install torch torchvision torchaudio statsmodels numpy pandas scikit-learn

import os, json, math, time, gc, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple
from scipy.stats import norm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from statsmodels.tsa.statespace.sarimax import SARIMAX

# -----------------------------
# Config
# -----------------------------
PREP_DIR   = "/content/drive/MyDrive/Close_res/prepared"
OUT_DIR    = "/content/drive/MyDrive/Close_res//models"
INDEXES    = ["SP500", "EUROSTOXX50", "NIKKEI225"]
SCENARIOS  = ["A","B","C","D"]
LOOKBACKS  = [2,3,5,10,20]
TARGET_COL = "y_close_t+1"     # keep target as price in files; we'll make z-target on the fly
BATCH_SIZE = 64
EPOCHS     = 40
LR         = 2e-3
PATIENCE   = 10
SEED       = 42
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

# -----------------------------
# IO helpers
# -----------------------------
def load_split(idx: str, sc: str, split: str) -> pd.DataFrame:
    path = f"{PREP_DIR}/{idx}/scenario_{sc}/{split}.csv.gz"
    df = pd.read_csv(path, parse_dates=[0], index_col=0).sort_index()
    return df

def load_roll(idx: str, sc: str) -> pd.DataFrame:
    path = f"{PREP_DIR}/{idx}/scenario_{sc}/rolling_stats.csv.gz"
    rs = pd.read_csv(path, parse_dates=[0], index_col=0).sort_index()
    return rs

def reconstruct_raw_close(df_split: pd.DataFrame, roll: pd.DataFrame) -> pd.Series:
    z_close = df_split["Close"]
    mu = roll.loc[df_split.index, "mu_Close"]
    sd = roll.loc[df_split.index, "sd_Close"]
    return z_close * sd + mu

def feature_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c not in ["y_close_t+1", "y_logret_t+1"]]

def metrics(y_true, y_pred, last_close=None) -> dict:
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.where(y_true == 0, np.nan, np.abs(y_true))
    smape = float(100.0 * np.nanmean(2.0*np.abs(y_true - y_pred) / (np.abs(y_true)+np.abs(y_pred)+1e-12)))
    out = {"MAE": mae, "RMSE": rmse, "sMAPE": smape}
    if last_close is not None:
        true_dir = np.sign(y_true - last_close); pred_dir = np.sign(y_pred - last_close)
        out["DIR_ACC"] = float((true_dir == pred_dir).mean())
    return out

# -----------------------------
# Dataset that uses z-target
# -----------------------------
class SeqDataset(Dataset):
    """
    For each row i (date t_i), build X = [i-L+1 .. i] z-features,
    target = z-target for t_{i+1} scaled with (mu_ti, sd_ti),
    and keep (mu_ti, sd_ti, last_close_raw_ti, y_raw_{t+1}) for inverse-transform & metrics.
    """
    def __init__(self, df: pd.DataFrame, roll: pd.DataFrame, lookback: int, target_col: str):
        self.df   = df.copy()
        self.roll = roll.loc[df.index]
        self.L = lookback
        self.target_col = target_col
        self.features = feature_columns(df)

        # arrays
        self.mu = self.roll["mu_Close"].astype(np.float32).values
        self.sd = self.roll["sd_Close"].astype(np.float32).values
        self.last_close_raw = reconstruct_raw_close(self.df, self.roll).astype(np.float32).values
        self.y_raw = self.df[self.target_col].astype(np.float32).values

        self.X, self.y_z, self.mu_t, self.sd_t, self.last_t, self.y_raw_t1 = self._build()

    def _build(self):
        Xs, yz, mu, sd, lastc, yraw = [], [], [], [], [], []
        vals = self.df[self.features].values.astype(np.float32)

        # avoid 0/NaN sd
        sd_safe = np.where(np.isfinite(self.sd) & (self.sd!=0), self.sd, np.nan)

        for i in range(self.L-1, len(self.df)):
            # features window [i-L+1 .. i]
            Xs.append(vals[i-self.L+1:i+1, :])
            # z-target uses (mu_t, sd_t) at time i
            mu_i = self.mu[i]; sd_i = sd_safe[i]
            if not np.isfinite(sd_i):
                continue
            yz.append( (self.y_raw[i] - mu_i) / sd_i )
            mu.append(mu_i); sd.append(sd_i)
            lastc.append(self.last_close_raw[i])
            yraw.append(self.y_raw[i])

        return (np.array(Xs, dtype=np.float32),
                np.array(yz, dtype=np.float32).reshape(-1,1),
                np.array(mu, dtype=np.float32).reshape(-1,1),
                np.array(sd, dtype=np.float32).reshape(-1,1),
                np.array(lastc, dtype=np.float32).reshape(-1,1),
                np.array(yraw, dtype=np.float32).reshape(-1,1))

    def __len__(self): return len(self.y_z)
    def __getitem__(self, idx):
        return (self.X[idx], self.y_z[idx], self.mu_t[idx], self.sd_t[idx], self.last_t[idx], self.y_raw_t1[idx])

# -----------------------------
# Models (PyTorch)
# -----------------------------
class LSTMModel(nn.Module):
    def __init__(self, n_feats, h1=96, h2=64, drop=0.2):
        super().__init__()
        self.l1 = nn.LSTM(n_feats, h1, batch_first=True)
        self.do1 = nn.Dropout(drop)
        self.l2 = nn.LSTM(h1, h2, batch_first=True)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(h2, 32), nn.ReLU(), nn.Linear(32,1))
    def forward(self, x):  # [B,L,F]
        x, _ = self.l1(x); x = self.do1(x)
        x, _ = self.l2(x)
        x = x[:, -1, :]
        return self.head(x)

class BiLSTMModel(nn.Module):
    def __init__(self, n_feats, h1=96, h2=64, drop=0.2):
        super().__init__()
        self.bi1 = nn.LSTM(n_feats, h1, batch_first=True, bidirectional=True)
        self.do1 = nn.Dropout(drop)
        self.bi2 = nn.LSTM(2*h1, h2, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(2*h2, 32), nn.ReLU(), nn.Linear(32,1))
    def forward(self, x):
        x, _ = self.bi1(x); x = self.do1(x)
        x, _ = self.bi2(x)
        x = x[:, -1, :]
        return self.head(x)

class TCNBlock(nn.Module):
    def __init__(self, ch, dil):
        super().__init__()
        pad = dil
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, 3, padding=pad, dilation=dil),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(ch, ch, 3, padding=pad, dilation=dil),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
    def forward(self, x): return x + self.net(x)

class TCN(nn.Module):
    def __init__(self, n_feats):
        super().__init__()
        self.proj = nn.Conv1d(n_feats, 64, 1)
        self.stack = nn.Sequential(*[TCNBlock(64,d) for d in [1,2,4,8,16,32]])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1))
    def forward(self, x):  # x: [B,L,F]
        x = x.transpose(1,2)
        x = self.proj(x)
        x = self.stack(x)
        return self.head(x)

# -----------------------------
# Training / inference
# -----------------------------
def make_loaders(df_tr, df_va, df_te, roll, L, target_col):
    ds_tr = SeqDataset(df_tr, roll, L, target_col)
    ds_va = SeqDataset(df_va, roll, L, target_col)
    ds_te = SeqDataset(df_te, roll, L, target_col)
    n_feats = len(feature_columns(df_tr))
    loaders = {
        "train": DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, drop_last=False),
        "val":   DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
        "test":  DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
    }
    return ds_tr, ds_va, ds_te, n_feats, loaders

def build_model(name, n_feats):
    if name=="LSTM":   return LSTMModel(n_feats).to(DEVICE)
    if name=="BiLSTM": return BiLSTMModel(n_feats).to(DEVICE)
    if name=="TCN":    return TCN(n_feats).to(DEVICE)
    raise ValueError(name)

def train_one(model, loaders, max_epochs=EPOCHS, lr=LR, patience=PATIENCE):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.SmoothL1Loss()  # Huber on z-target
    best_val = np.inf; bad=0; best_state=None

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))

    for ep in range(1, max_epochs+1):
        # ---- train ----
        model.train(); train_loss = 0.0
        for xb, yb_z, _, _, _, _ in loaders["train"]:
            xb, yb_z = xb.to(DEVICE), yb_z.to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                pred_z = model(xb)
                loss = loss_fn(pred_z, yb_z)
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            train_loss += loss.item() * len(xb)
        train_loss /= max(1,len(loaders["train"].dataset))

        # ---- val ----
        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for xb, yb_z, _, _, _, _ in loaders["val"]:
                xb, yb_z = xb.to(DEVICE), yb_z.to(DEVICE)
                pred_z = model(xb)
                loss = loss_fn(pred_z, yb_z)
                val_loss += loss.item() * len(xb)
        val_loss /= max(1,len(loaders["val"].dataset))

        print(f"  epoch {ep:02d} | z-train {train_loss:.4f} | z-val {val_loss:.4f}", end="\r")
        if val_loss + 1e-9 < best_val:
            best_val = val_loss; best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if bad >= patience:
                print(f"\n  early stop @ {ep} (best z-val={best_val:.4f})")
                break
    print("")
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def predict_raw(model, loader):
    model.eval()
    y_raw_all, yhat_raw_all, last_all = [], [], []
    with torch.no_grad():
        for xb, yb_z, mu, sd, lastc, yb_raw in loader:
            xb = xb.to(DEVICE)
            pred_z = model(xb).cpu().numpy().reshape(-1,1)
            mu = mu.numpy(); sd = sd.numpy()
            yhat_raw = mu + sd * pred_z
            y_raw_all.append(yb_raw.numpy().reshape(-1))
            yhat_raw_all.append(yhat_raw.reshape(-1))
            last_all.append(lastc.numpy().reshape(-1))
    return np.concatenate(y_raw_all), np.concatenate(yhat_raw_all), np.concatenate(last_all)

# -----------------------------
# Baselines
# -----------------------------
def baseline_naive(last_close_raw: np.ndarray):
    # ŷ_{t+1} = Close_t  (already raw)
    return last_close_raw

def baseline_arima(train_series: pd.Series, full_series: pd.Series, forecast_idx: pd.Index):
    # One-step expanding forecast over (val+test)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = SARIMAX(train_series, order=(1,1,0),
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    preds = []
    for t in forecast_idx:
        f = res.get_forecast(steps=1)
        preds.append(float(f.predicted_mean.iloc[0]))
        res = res.append(endog=[full_series.loc[t]], refit=False)
    return np.array(preds, dtype=np.float32)

# -----------------------------
# Runner
# -----------------------------
results_rows = []

for idx in INDEXES:
    print(f"\n=== INDEX: {idx} ===")
    # Baselines from Scenario A scaling
    df_tr_a = load_split(idx, "A", "train")
    df_va_a = load_split(idx, "A", "val")
    df_te_a = load_split(idx, "A", "test")
    roll_a  = load_roll(idx, "A")

    # Raw Close_t for each split (to score baselines)
    last_val_raw = reconstruct_raw_close(df_va_a, roll_a).values
    last_tes_raw = reconstruct_raw_close(df_te_a, roll_a).values
    y_val = df_va_a[TARGET_COL].values
    y_tes = df_te_a[TARGET_COL].values

    # Naïve
    naive_val = baseline_naive(last_val_raw); naive_tes = baseline_naive(last_tes_raw)
    m_val = metrics(y_val, naive_val, last_val_raw); m_tes = metrics(y_tes, naive_tes, last_tes_raw)
    results_rows.append({"Index": idx, "Scenario": "BASE", "Model": "NaiveRW", "Lookback": 1, "Split":"val", **m_val})
    results_rows.append({"Index": idx, "Scenario": "BASE", "Model": "NaiveRW", "Lookback": 1, "Split":"test", **m_tes})
    print(f"  Baseline NaiveRW  | val MAE={m_val['MAE']:.2f} | test MAE={m_tes['MAE']:.2f}")

    # ARIMA: fit on (train), forecast val+test one-step
    full_a   = pd.concat([df_tr_a, df_va_a, df_te_a], axis=0)
    full_roll= roll_a.loc[full_a.index]
    full_close_raw = reconstruct_raw_close(full_a, full_roll)
    n_tr, n_va = len(df_tr_a), len(df_va_a)
    idx_all = full_a.index
    arima_preds = baseline_arima(full_close_raw.iloc[:n_tr], full_close_raw, idx_all[n_tr:])
    arima_val, arima_tes = arima_preds[:n_va], arima_preds[n_va:]
    m_val = metrics(y_val, arima_val, last_val_raw); m_tes = metrics(y_tes, arima_tes, last_tes_raw)
    results_rows.append({"Index": idx, "Scenario": "BASE", "Model": "ARIMA(1,1,0)", "Lookback": 1, "Split":"val", **m_val})
    results_rows.append({"Index": idx, "Scenario": "BASE", "Model": "ARIMA(1,1,0)", "Lookback": 1, "Split":"test", **m_tes})
    print(f"  Baseline ARIMA    | val MAE={m_val['MAE']:.2f} | test MAE={m_tes['MAE']:.2f}")

    # Deep models
    for sc in SCENARIOS:
        print(f"\n  -- Scenario {sc} --")
        df_tr = load_split(idx, sc, "train")
        df_va = load_split(idx, sc, "val")
        df_te = load_split(idx, sc, "test")
        roll  = load_roll(idx, sc)

        for L in LOOKBACKS:
            ds_tr, ds_va, ds_te, n_feats, loaders = make_loaders(df_tr, df_va, df_te, roll, L, TARGET_COL)
            if len(ds_va)==0 or len(ds_te)==0:
                print(f"    lookback={L}: insufficient rows; skipping")
                continue

            for model_name in ["LSTM", "BiLSTM", "TCN"]:
                run_id = f"{idx}_{sc}_{model_name}_L{L}"
                print(f"    [{run_id}] training... (features={n_feats}, train={len(ds_tr)}, val={len(ds_va)}, test={len(ds_te)})")
                model = build_model(model_name, n_feats)
                model = train_one(model, loaders, max_epochs=EPOCHS, lr=LR, patience=PATIENCE)

                # Evaluate in RAW space (invert z with mu_t, sd_t)
                yv_raw, pv_raw, lv_raw = predict_raw(model, loaders["val"])
                yt_raw, pt_raw, lt_raw = predict_raw(model, loaders["test"])
                mv = metrics(yv_raw, pv_raw, lv_raw); mt = metrics(yt_raw, pt_raw, lt_raw)

                # Save predictions & model
                save_dir = os.path.join(OUT_DIR, idx, sc, model_name, f"L{L}")
                os.makedirs(save_dir, exist_ok=True)
                np.save(os.path.join(save_dir, "y_val.npy"), yv_raw)
                np.save(os.path.join(save_dir, "yhat_val.npy"), pv_raw)   # RAW
                np.save(os.path.join(save_dir, "last_close_val.npy"), lv_raw)
                np.save(os.path.join(save_dir, "y_test.npy"), yt_raw)
                np.save(os.path.join(save_dir, "yhat_test.npy"), pt_raw)  # RAW
                np.save(os.path.join(save_dir, "last_close_test.npy"), lt_raw)
                torch.save(model.state_dict(), os.path.join(save_dir, "model.pt"))

                results_rows.append({"Index": idx, "Scenario": sc, "Model": model_name, "Lookback": L, "Split":"val", **mv})
                results_rows.append({"Index": idx, "Scenario": sc, "Model": model_name, "Lookback": L, "Split":"test", **mt})
                print(f"      -> val MAE={mv['MAE']:.2f} | test MAE={mt['MAE']:.2f} | DIR(test)={mt.get('DIR_ACC', np.nan):.3f}")

# Save summary
summary = pd.DataFrame(results_rows)
summary_path = os.path.join(OUT_DIR, "results_summary.csv")
summary.to_csv(summary_path, index=False)
print(f"\nSaved results summary: {summary_path}")



=== INDEX: SP500 ===
  Baseline NaiveRW  | val MAE=36.03 | test MAE=35.66


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


  Baseline ARIMA    | val MAE=52.30 | test MAE=52.54

  -- Scenario A --
    [SP500_A_LSTM_L2] training... (features=1, train=2686, val=383, test=767)


/tmp/ipython-input-2645554607.py:207: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipython-input-2645554607.py:215: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):


  epoch 24 | z-train 0.0958 | z-val 0.0925
  early stop @ 24 (best z-val=0.0922)

      -> val MAE=36.83 | test MAE=35.67 | DIR(test)=0.529
    [SP500_A_BiLSTM_L2] training... (features=1, train=2686, val=383, test=767)
  epoch 32 | z-train 0.0949 | z-val 0.0944
  early stop @ 32 (best z-val=0.0916)

      -> val MAE=36.45 | test MAE=35.67 | DIR(test)=0.537
    [SP500_A_TCN_L2] training... (features=1, train=2686, val=383, test=767)
  epoch 18 | z-train 0.0965 | z-val 0.0991
  early stop @ 18 (best z-val=0.0908)

      -> val MAE=36.19 | test MAE=36.00 | DIR(test)=0.501
    [SP500_A_LSTM_L3] training... (features=1, train=2685, val=382, test=766)


...

  epoch 40 | z-train 0.0889 | z-val 0.1235
      -> val MAE=280.20 | test MAE=330.57 | DIR(test)=0.513
    [NIKKEI225_D_BiLSTM_L5] training... (features=7, train=2607, val=369, test=743)
  epoch 30 | z-train 0.0944 | z-val 0.1199
  early stop @ 30 (best z-val=0.1190)

      -> val MAE=278.85 | test MAE=331.84 | DIR(test)=0.498
    [NIKKEI225_D_TCN_L5] training... (features=7, train=2607, val=369, test=743)
  epoch 24 | z-train 0.0868 | z-val 0.1292
  early stop @ 24 (best z-val=0.1203)

      -> val MAE=284.88 | test MAE=341.30 | DIR(test)=0.478
    [NIKKEI225_D_LSTM_L10] training... (features=7, train=2602, val=364, test=738)


/tmp/ipython-input-2645554607.py:207: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipython-input-2645554607.py:215: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):


  epoch 18 | z-train 0.0929 | z-val 0.1290
  early stop @ 18 (best z-val=0.1210)

      -> val MAE=280.23 | test MAE=335.71 | DIR(test)=0.481
    [NIKKEI225_D_BiLSTM_L10] training... (features=7, train=2602, val=364, test=738)
  epoch 31 | z-train 0.0918 | z-val 0.1258
  early stop @ 31 (best z-val=0.1185)

      -> val MAE=277.61 | test MAE=330.56 | DIR(test)=0.500
    [NIKKEI225_D_TCN_L10] training... (features=7, train=2602, val=364, test=738)
  epoch 27 | z-train 0.0852 | z-val 0.1376
  early stop @ 27 (best z-val=0.1237)

      -> val MAE=280.29 | test MAE=341.34 | DIR(test)=0.516
    [NIKKEI225_D_LSTM_L20] training... (features=7, train=2592, val=354, test=728)


/tmp/ipython-input-2645554607.py:207: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipython-input-2645554607.py:215: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):


  epoch 15 | z-train 0.0936 | z-val 0.1243
  early stop @ 15 (best z-val=0.1240)

      -> val MAE=281.72 | test MAE=351.75 | DIR(test)=0.478
    [NIKKEI225_D_BiLSTM_L20] training... (features=7, train=2592, val=354, test=728)
  epoch 24 | z-train 0.0898 | z-val 0.1295
  early stop @ 24 (best z-val=0.1203)

      -> val MAE=275.02 | test MAE=332.57 | DIR(test)=0.508
    [NIKKEI225_D_TCN_L20] training... (features=7, train=2592, val=354, test=728)
  epoch 26 | z-train 0.0875 | z-val 0.1399
  early stop @ 26 (best z-val=0.1271)

      -> val MAE=285.73 | test MAE=385.54 | DIR(test)=0.477

Saved results summary: /content/drive/MyDrive/Close_res//models/results_summary.csv


In [ ]:
# =========================================
# Post-training evaluation (top-journal pack)
# - MAE, RMSE, sMAPE, **MAPE**, MASE, RelMAE, RelRMSE, **RelMAPE**
# - Model selection by VAL-MASE (default; switchable)
# - Diebold–Mariano tests on TEST (A vs B/C/D)
# =========================================
!pip -q install numpy pandas scipy

import os, json, math, warnings
import numpy as np
import pandas as pd
from scipy.stats import norm

# ------------ Config ------------
PREP_DIR    = "/content/drive/MyDrive/Close_res/prepared"
MODELS_DIR  = "/content/drive/MyDrive/Close_res/models"
OUT_DIR     = "/content/drive/MyDrive/Close_res/models"
INDEXES     = ["SP500", "EUROSTOXX50", "NIKKEI225"]
SCENARIOS   = ["A","B","C","D"]
MODELS      = ["LSTM","BiLSTM","TCN"]
LOOKBACKS   = [2,3,5,10,20]
TARGET_COL  = "y_close_t+1"
CANONICAL_MODEL = "LSTM"   # for DM tests
INCLUDE_DIRACC = False      # set True if you want directional accuracy too

# Selection criterion for picking the lookback on VAL:
SELECT_BY = "MASE"          # options: "MASE" (recommended), or "MAPE" if you prefer

os.makedirs(OUT_DIR, exist_ok=True)

# ------------ Helpers ------------
def load_split(idx, sc, split):
    path = f"{PREP_DIR}/{idx}/scenario_{sc}/{split}.csv.gz"
    df = pd.read_csv(path, parse_dates=[0], index_col=0).sort_index()
    return df

def load_roll(idx, sc):
    path = f"{PREP_DIR}/{idx}/scenario_{sc}/rolling_stats.csv.gz"
    rs = pd.read_csv(path, parse_dates=[0], index_col=0).sort_index()
    return rs

def reconstruct_raw_close(df, roll):
    z = df["Close"]
    mu = roll.loc[df.index, "mu_Close"]
    sd = roll.loc[df.index, "sd_Close"]
    return z * sd + mu

def smape(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    num = 2.0 * np.abs(y - yhat)
    den = np.abs(y) + np.abs(yhat) + 1e-12
    return 100.0 * np.nanmean(num / den)

def mape(y, yhat):
    # Exact MAPE (no epsilon). Safe here because index levels y>0.
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    return 100.0 * np.mean(np.abs((y - yhat) / y))

def mae(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    return float(np.sqrt(np.mean((y - yhat)**2)))

def mase(y, yhat, denom):
    return float(mae(y, yhat) / (denom if denom > 0 else np.nan))

def diracc(y_true_next, y_pred_next, last_close):
    return float(np.mean(np.sign(y_true_next - last_close) == np.sign(y_pred_next - last_close)))

def dm_test(y, yhat1, yhat2, loss="ae", nw_lags="auto"):
    """
    Diebold–Mariano test with Bartlett-kernel Newey–West variance.
    loss='ae' (|e|) or 'se' (e^2).
    Returns (stat, pvalue, T, dbar).
    """
    y = np.asarray(y, float)
    e1 = y - np.asarray(yhat1, float)
    e2 = y - np.asarray(yhat2, float)
    if loss == "ae":
        L1, L2 = np.abs(e1), np.abs(e2)
    elif loss == "se":
        L1, L2 = e1**2, e2**2
    else:
        raise ValueError("loss must be 'ae' or 'se'")
    d = L1 - L2
    T = len(d)
    if T < 5:
        return np.nan, np.nan, T, np.nan

    dbar = np.mean(d)
    # Newey–West variance for the mean, Bartlett weights
    if nw_lags == "auto":
        L = int(np.floor(1.5 * (T ** (1/3))))
    else:
        L = int(nw_lags)

    d_center = d - dbar
    gamma0 = np.dot(d_center, d_center) / T
    s = gamma0
    for k in range(1, min(L, T-1) + 1):
        cov = np.dot(d_center[k:], d_center[:-k]) / T
        w = 1.0 - k/(L+1.0)
        s += 2.0 * w * cov
    var_dbar = s / T
    if var_dbar <= 0 or not np.isfinite(var_dbar):
        return np.nan, np.nan, T, dbar

    stat = dbar / np.sqrt(var_dbar)
    pval = 2.0 * (1.0 - norm.cdf(np.abs(stat)))
    return float(stat), float(pval), int(T), float(dbar)

def run_exists(idx, sc, model, L):
    path = f"{MODELS_DIR}/{idx}/{sc}/{model}/L{L}/y_test.npy"
    return os.path.exists(path)

def load_preds(idx, sc, model, L, split):
    base = f"{MODELS_DIR}/{idx}/{sc}/{model}/L{L}"
    y = np.load(f"{base}/y_{split}.npy")
    yhat = np.load(f"{base}/yhat_{split}.npy")
    last = np.load(f"{base}/last_close_{split}.npy")
    return y, yhat, last

# Recreate the exact sample dates used by the dataset (skip rows where rolling sd_t was invalid)
def dataset_sample_dates(df, roll, L):
    sd = roll.loc[df.index, "sd_Close"].values.astype(float)
    valid = np.isfinite(sd) & (sd != 0)
    idx = df.index
    out = []
    for i in range(L-1, len(df)):
        if valid[i]:
            out.append(idx[i])
    return pd.Index(out)

# ------------ Evaluate all runs ------------
rows = []

for idx in INDEXES:
    for sc in SCENARIOS:
        # Train split for MASE denominator
        df_tr = load_split(idx, sc, "train")
        roll_tr = load_roll(idx, sc)
        train_last_raw = reconstruct_raw_close(df_tr, roll_tr).values
        train_y = df_tr[TARGET_COL].values
        mase_denom = np.mean(np.abs(train_y - train_last_raw))  # in-sample naïve MAE

        for model in MODELS:
            for L in LOOKBACKS:
                if not run_exists(idx, sc, model, L):
                    continue

                # ---------- VAL ----------
                df_va = load_split(idx, sc, "val"); roll_va = load_roll(idx, sc)
                yv, pv, lv = load_preds(idx, sc, model, L, "val")
                naive_va = lv  # last_close_raw already saved
                mae_naive_va = mae(yv, naive_va)
                rmse_naive_va = rmse(yv, naive_va)
                mape_naive_va = mape(yv, naive_va)

                r = {
                    "Index": idx, "Scenario": sc, "Model": model, "Lookback": L, "Split": "val",
                    "MAE": mae(yv, pv),
                    "RMSE": rmse(yv, pv),
                    "sMAPE": smape(yv, pv),
                    "MAPE": mape(yv, pv),
                    "MASE": mase(yv, pv, mase_denom),
                    "RelMAE": mae(yv, pv) / (mae_naive_va if mae_naive_va>0 else np.nan),
                    "RelRMSE": rmse(yv, pv) / (rmse_naive_va if rmse_naive_va>0 else np.nan),
                    "RelMAPE": mape(yv, pv) / (mape_naive_va if mape_naive_va>0 else np.nan),
                }
                if INCLUDE_DIRACC:
                    r["DIR_ACC"] = diracc(yv, pv, lv)
                rows.append(r)

                # ---------- TEST ----------
                df_te = load_split(idx, sc, "test"); roll_te = load_roll(idx, sc)
                yt, pt, lt = load_preds(idx, sc, model, L, "test")
                naive_te = lt
                mae_naive_te = mae(yt, naive_te)
                rmse_naive_te = rmse(yt, naive_te)
                mape_naive_te = mape(yt, naive_te)

                r = {
                    "Index": idx, "Scenario": sc, "Model": model, "Lookback": L, "Split": "test",
                    "MAE": mae(yt, pt),
                    "RMSE": rmse(yt, pt),
                    "sMAPE": smape(yt, pt),
                    "MAPE": mape(yt, pt),
                    "MASE": mase(yt, pt, mase_denom),
                    "RelMAE": mae(yt, pt) / (mae_naive_te if mae_naive_te>0 else np.nan),
                    "RelRMSE": rmse(yt, pt) / (rmse_naive_te if rmse_naive_te>0 else np.nan),
                    "RelMAPE": mape(yt, pt) / (mape_naive_te if mape_naive_te>0 else np.nan),
                }
                if INCLUDE_DIRACC:
                    r["DIR_ACC"] = diracc(yt, pt, lt)
                rows.append(r)

results = pd.DataFrame(rows)
results_path = f"{OUT_DIR}/results_topjournals.csv"
results.to_csv(results_path, index=False)
print(f"Saved all-run metrics -> {results_path}")

# ------------ Model selection (VAL-<SELECT_BY>) ------------
sel_rows = []
metric_primary = SELECT_BY  # "MASE" (default) or "MAPE"
fallback_order = ["MAE","RMSE"]  # tie-breakers

for (idx, sc, model), sub in results[results["Split"]=="val"].groupby(["Index","Scenario","Model"]):
    sub = sub.copy()
    sort_cols = [metric_primary] + fallback_order
    best = sub.sort_values(sort_cols, ascending=True).iloc[0]
    best_L = int(best["Lookback"])
    # fetch corresponding TEST row
    test_row = results[(results["Index"]==idx)&(results["Scenario"]==sc)&
                       (results["Model"]==model)&(results["Lookback"]==best_L)&
                       (results["Split"]=="test")].iloc[0].to_dict()
    test_row["SelBy"] = f"VAL-{metric_primary}"
    sel_rows.append(test_row)

sel = pd.DataFrame(sel_rows)
sel_path = f"{OUT_DIR}/selection_topjournals.csv"
sel.to_csv(sel_path, index=False)
print(f"Saved selection (VAL-{metric_primary}) -> {sel_path}")

# ------------ DM tests: A vs (B,C,D) on TEST for CANONICAL_MODEL ------------
def load_run_bundle(idx, sc, model, L):
    # Build the exact date index used by this run, then align arrays to it
    df_te = load_split(idx, sc, "test"); roll_te = load_roll(idx, sc)
    dates = dataset_sample_dates(df_te, roll_te, L)
    y, yhat, last = load_preds(idx, sc, model, L, "test")
    n = min(len(dates), len(y))  # safety, should match
    return pd.DataFrame({"y": y[:n].reshape(-1), "yhat": yhat[:n].reshape(-1), "last": last[:n].reshape(-1)},
                        index=dates[:n])

dm_rows = []
pairs = [("A","B"),("A","C"),("A","D")]

for idx in INDEXES:
    # pick best lookback for each scenario under the canonical model
    best_L_for = {}
    for sc in SCENARIOS:
        sub = sel[(sel["Index"]==idx)&(sel["Scenario"]==sc)&(sel["Model"]==CANONICAL_MODEL)]
        best_L_for[sc] = int(sub.iloc[0]["Lookback"]) if len(sub) else None

    for scA, scB in pairs:
        LA, LB = best_L_for.get(scA), best_L_for.get(scB)
        if (LA is None) or (LB is None):
            continue
        if not (run_exists(idx, scA, CANONICAL_MODEL, LA) and run_exists(idx, scB, CANONICAL_MODEL, LB)):
            continue

        A = load_run_bundle(idx, scA, CANONICAL_MODEL, LA)
        B = load_run_bundle(idx, scB, CANONICAL_MODEL, LB)

        # align by common dates (scenarios can have different test indices)
        common = A.index.intersection(B.index)
        A = A.loc[common]; B = B.loc[common]
        if len(common) < 20:
            continue

        # DM test on absolute error (primary) and squared error (optional)
        stat_ae, p_ae, T, dbar_ae = dm_test(A["y"].values, A["yhat"].values, B["yhat"].values, loss="ae", nw_lags="auto")
        stat_se, p_se, _,  dbar_se = dm_test(A["y"].values, A["yhat"].values, B["yhat"].values, loss="se", nw_lags="auto")

        # Decide winner by MAE on aligned set
        maeA = mae(A["y"], A["yhat"]); maeB = mae(B["y"], B["yhat"])
        winner = scA if maeA < maeB else scB

        dm_rows.append({
            "Index": idx,
            "Model": CANONICAL_MODEL,
            "Pair": f"{scA} vs {scB}",
            "Lookback_A": LA, "Lookback_B": LB,
            "Aligned_T": T,
            "MAE_A": maeA, "MAE_B": maeB,
            "DM_stat_AE": stat_ae, "p_AE": p_ae, "dbar_AE": dbar_ae,
            "DM_stat_SE": stat_se, "p_SE": p_se, "dbar_SE": dbar_se,
            "Winner_by_MAE": winner
        })

dm = pd.DataFrame(dm_rows)
dm_path = f"{OUT_DIR}/dm_tests.csv"
dm.to_csv(dm_path, index=False)
print(f"Saved DM tests -> {dm_path}")

# ---- example printouts (first few rows) ----
print("\n== sample: results_topjournals.csv ==")
print(results.head(5).to_string(index=False))
print("\n== sample: selection_topjournals.csv ==")
print(sel.head(5).to_string(index=False))
print("\n== sample: dm_tests.csv ==")
print(dm.head(5).to_string(index=False))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Close_res/prepared/SP500/scenario_A/train.csv.gz'

In [ ]:
import os
import pandas as pd

# ------------ Config ------------
PREP_DIR    = "/content/drive/MyDrive/Close_res/prepared"
INDEXES     = ["SP500", "EUROSTOXX50", "NIKKEI225"]
SCENARIOS   = ["A","B","C","D"]
TARGET_COL  = "y_close_t+1"

# ------------ Helper ------------
def load_split(idx, sc, split):
    path = f"{PREP_DIR}/{idx}/scenario_{sc}/{split}.csv.gz"
    df = pd.read_csv(path, parse_dates=[0], index_col=0).sort_index()
    return df

# ------------ Print features for each scenario ------------
print("\n" + "="*60)
print("FEATURES BY INDEX AND SCENARIO")
print("="*60)

for idx in INDEXES:
    for sc in SCENARIOS:
        try:
            df = load_split(idx, sc, "train")
            features = [col for col in df.columns if col != TARGET_COL]
            print(f"\n{idx} - Scenario {sc}:")
            print(f"  Number of features: {len(features)}")
            print(f"  Features: {features}")
        except Exception as e:
            print(f"\n{idx} - Scenario {sc}: Could not load - {e}")


FEATURES BY INDEX AND SCENARIO

SP500 - Scenario A:
  Number of features: 2
  Features: ['Close', 'y_logret_t+1']

SP500 - Scenario B:
  Number of features: 6
  Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'y_logret_t+1']

SP500 - Scenario C:
  Number of features: 25
  Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'sma_5', 'sma_20', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_hist', 'rsi_14', 'stoch_k_14_3', 'stoch_d_14_3', 'bb_m_20_2', 'bb_h_20_2', 'bb_l_20_2', 'bb_bw_20_2', 'bb_pctb_20_2', 'adx_14', 'obv', 'roc_10', 'atr_14', 'y_logret_t+1']

SP500 - Scenario D:
  Number of features: 8
  Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'roc_10', 'rsi_14', 'y_logret_t+1']

EUROSTOXX50 - Scenario A:
  Number of features: 2
  Features: ['Close', 'y_logret_t+1']

EUROSTOXX50 - Scenario B:
  Number of features: 6
  Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'y_logret_t+1']

EUROSTOXX50 - Scenario C:
  Number of features: 25
  Features: ['Open', 'High', '

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import norm

# ------------ Config ------------
PREP_DIR    = "/content/drive/MyDrive/Close_res/prepared"
MODELS_DIR  = "/content/drive/MyDrive/Close_res/models"
OUT_DIR     = "/content/drive/MyDrive/Close_res/models"
INDEXES     = ["SP500", "EUROSTOXX50", "NIKKEI225"]
SCENARIOS   = ["A","B","C","D"]
MODELS      = ["LSTM","BiLSTM","TCN"]

# Load your existing selection file (already computed)
sel = pd.read_csv(f"{OUT_DIR}/selection_topjournals.csv")

# ------------ Helper functions (only what's needed) ------------
def load_split(idx, sc, split):
    path = f"{PREP_DIR}/{idx}/scenario_{sc}/{split}.csv.gz"
    return pd.read_csv(path, parse_dates=[0], index_col=0).sort_index()

def load_roll(idx, sc):
    path = f"{PREP_DIR}/{idx}/scenario_{sc}/rolling_stats.csv.gz"
    return pd.read_csv(path, parse_dates=[0], index_col=0).sort_index()

def run_exists(idx, sc, model, L):
    return os.path.exists(f"{MODELS_DIR}/{idx}/{sc}/{model}/L{L}/y_test.npy")

def load_preds(idx, sc, model, L, split):
    base = f"{MODELS_DIR}/{idx}/{sc}/{model}/L{L}"
    return np.load(f"{base}/y_{split}.npy"), np.load(f"{base}/yhat_{split}.npy"), np.load(f"{base}/last_close_{split}.npy")

def dataset_sample_dates(df, roll, L):
    sd = roll.loc[df.index, "sd_Close"].values.astype(float)
    valid = np.isfinite(sd) & (sd != 0)
    return pd.Index([df.index[i] for i in range(L-1, len(df)) if valid[i]])

def load_run_bundle(idx, sc, model, L):
    df_te = load_split(idx, sc, "test"); roll_te = load_roll(idx, sc)
    dates = dataset_sample_dates(df_te, roll_te, L)
    y, yhat, last = load_preds(idx, sc, model, L, "test")
    n = min(len(dates), len(y))
    return pd.DataFrame({"y": y[:n].reshape(-1), "yhat": yhat[:n].reshape(-1)}, index=dates[:n])

def mae(y, yhat):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))

def dm_test(y, yhat1, yhat2, loss="ae"):
    y, e1, e2 = np.asarray(y), y - np.asarray(yhat1), y - np.asarray(yhat2)
    L1, L2 = (np.abs(e1), np.abs(e2)) if loss == "ae" else (e1**2, e2**2)
    d = L1 - L2; T = len(d); dbar = np.mean(d)
    if T < 5: return np.nan, np.nan, T, np.nan
    L = int(np.floor(1.5 * (T ** (1/3))))
    d_center = d - dbar
    s = np.dot(d_center, d_center) / T
    for k in range(1, min(L, T-1) + 1):
        s += 2.0 * (1.0 - k/(L+1.0)) * np.dot(d_center[k:], d_center[:-k]) / T
    var_dbar = s / T
    if var_dbar <= 0: return np.nan, np.nan, T, dbar
    stat = dbar / np.sqrt(var_dbar)
    return float(stat), float(2.0 * (1.0 - norm.cdf(np.abs(stat)))), int(T), float(dbar)

# ------------ DM tests for ALL models ------------
dm_rows = []
pairs = [("A","B"), ("A","C")]  # Ignoring D as you requested

for idx in INDEXES:
    for model in MODELS:
        best_L = {}
        for sc in SCENARIOS:
            sub = sel[(sel["Index"]==idx) & (sel["Scenario"]==sc) & (sel["Model"]==model)]
            best_L[sc] = int(sub.iloc[0]["Lookback"]) if len(sub) else None

        for scA, scB in pairs:
            LA, LB = best_L.get(scA), best_L.get(scB)
            if LA is None or LB is None: continue
            if not (run_exists(idx, scA, model, LA) and run_exists(idx, scB, model, LB)): continue

            A = load_run_bundle(idx, scA, model, LA)
            B = load_run_bundle(idx, scB, model, LB)
            common = A.index.intersection(B.index)
            A, B = A.loc[common], B.loc[common]
            if len(common) < 20: continue

            stat_ae, p_ae, T, dbar_ae = dm_test(A["y"].values, A["yhat"].values, B["yhat"].values, "ae")
            stat_se, p_se, _, dbar_se = dm_test(A["y"].values, A["yhat"].values, B["yhat"].values, "se")
            maeA, maeB = mae(A["y"], A["yhat"]), mae(B["y"], B["yhat"])

            dm_rows.append({
                "Index": idx, "Model": model, "Pair": f"{scA} vs {scB}",
                "Lookback_A": LA, "Lookback_B": LB, "Aligned_T": T,
                "MAE_A": maeA, "MAE_B": maeB,
                "DM_stat_AE": stat_ae, "p_AE": p_ae, "dbar_AE": dbar_ae,
                "DM_stat_SE": stat_se, "p_SE": p_se, "dbar_SE": dbar_se,
                "Winner_by_MAE": scA if maeA < maeB else scB
            })

dm = pd.DataFrame(dm_rows)
dm.to_csv(f"{OUT_DIR}/dm_tests_all_models.csv", index=False)
print(dm.to_string(index=False))

      Index  Model   Pair  Lookback_A  Lookback_B  Aligned_T      MAE_A      MAE_B  DM_stat_AE         p_AE    dbar_AE  DM_stat_SE     p_SE       dbar_SE Winner_by_MAE
      SP500   LSTM A vs B          20           2        749  35.553951  36.243481   -1.691234 9.079217e-02  -0.689527   -0.695107 0.486988    -41.712097             A
      SP500   LSTM A vs C          20           3        749  35.553951  37.556465   -3.659492 2.527160e-04  -2.002513   -2.628705 0.008571   -213.177200             A
      SP500 BiLSTM A vs B           2          10        759  35.767365  36.125290   -0.683302 4.944162e-01  -0.357927    0.391471 0.695449     54.306591             A
      SP500 BiLSTM A vs C           2           2        767  35.674809  37.379433   -2.741592 6.114220e-03  -1.704622   -2.811964 0.004924   -234.274689             A
      SP500    TCN A vs B           2           2        767  36.003227  36.057167   -0.128171 8.980139e-01  -0.053940   -0.375204 0.707509    -20.884949       

In [ ]:
!pip -q install --no-deps yfinance==0.2.41 ta==0.11.0
